# TaxGPT — Episode 2: Tokenization (BPE)

Companion notebook to blog post *"Tokenization Explained: Why LLMs Don't Read Words (TaxGPT Episode 2)"*.

TaxGPT uses **GPT-2's Byte Pair Encoding (BPE) tokenizer** (vocab size 50,257) rather than a custom-trained tokenizer — see the blog post for why that's a deliberate v1 tradeoff. This notebook uses `tiktoken` (the same BPE implementation family) to show exactly what that tokenizer does to real GST-style legal text.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 2. Andrej Karpathy, *Let's Build the GPT Tokenizer* (`karpathy/minbpe`).

> **Run note:** the first `tiktoken.get_encoding("gpt2")` call downloads the GPT-2 vocab/merges files the first time it runs on a machine, and caches them locally after that. It needs outbound internet access once (works fine on a normal laptop, Colab, or Kaggle). If you're on a locked-down/offline environment, install `transformers` instead and swap in `GPT2TokenizerFast.from_pretrained("gpt2")`, or point `tiktoken` at a pre-downloaded cache via the `TIKTOKEN_CACHE_DIR` environment variable.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print("GPT-2 BPE vocab size:", enc.n_vocab)

## 1. Encoding a GST sentence — words vs. tokens

Notice this is **not** one-token-per-word. Compound and domain-specific terms get split into subword pieces.

In [ ]:
sentence = "Input Tax Credit shall not be available in respect of goods used for personal consumption under Section 16(4)."

token_ids = enc.encode(sentence)
tokens_str = [enc.decode([t]) for t in token_ids]

print(f"Characters: {len(sentence)}")
print(f"Words (naive split): {len(sentence.split())}")
print(f"BPE tokens: {len(token_ids)}")
print()
for tid, tstr in zip(token_ids, tokens_str):
    print(f"{tid:>6}  ->  {tstr!r}")

## 2. Domain terms get split — this is the tradeoff the blog post flags

Watch what happens to GST-specific vocabulary that GPT-2's tokenizer was never trained on.

In [ ]:
domain_terms = ["HSN", "e-way bill", "composition scheme", "ITC", "Input Tax Credit", "input-tax-credit", "reverse charge mechanism"]

for term in domain_terms:
    ids = enc.encode(term)
    pieces = [enc.decode([t]) for t in ids]
    print(f"{term!r:<28} -> {len(ids)} tokens -> {pieces}")

Notice `"ITC"`, `"Input Tax Credit"`, and `"input-tax-credit"` — semantically the same concept — encode to completely different token sequences and different lengths. A GST-specific tokenizer trained on TaxGPT's own corpus would very likely collapse these into fewer, more consistent tokens. That's the "legitimate next iteration" the blog post mentions.

## 3. Reproducing the real corpus numbers

The actual TaxGPT corpus: 2,388 documents, 10,621,967 tokens after BPE encoding, packed into 1,024-token sequences (9,341 train + 1,033 val). Here's the packing arithmetic on a toy passage, so the shapes make sense before scaling up.

In [ ]:
toy_corpus_ids = enc.encode(sentence * 50)  # stand-in for a big corpus
context_length = 1024

n_full_sequences = len(toy_corpus_ids) // context_length
print(f"toy corpus tokens: {len(toy_corpus_ids)}")
print(f"context length: {context_length}")
print(f"full training sequences this toy corpus would yield: {n_full_sequences}")

# Real TaxGPT numbers, for reference:
real_total_tokens = 10_621_967
real_train_seqs = 9_341
real_val_seqs = 1_033
print()
print(f"Real TaxGPT corpus: {real_total_tokens:,} tokens")
print(f"Real TaxGPT: {real_train_seqs:,} train sequences + {real_val_seqs:,} val sequences @ {context_length} tokens each")
print(f"Sanity check (approx): {(real_train_seqs + real_val_seqs) * context_length:,} tokens packed")

## Takeaway

Tokenized integers are still just IDs — nothing here tells the model that token 4092 is "close to" token 1187. That relationship is *learned*, via embeddings.

**Next notebook: Episode 3 — Token embeddings and positional embeddings.**